# 03 - Load FX Rates

Loads the simulated FX reference data into a Delta table. The load is
idempotent because it merges by `(rate_date, currency)`.

In [ ]:
import re
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text(
    "landing_root",
    "/Volumes/abc_retail/dev/landing",
    "Landing root",
)
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = dbutils.widgets.get("catalog").strip()
landing_root = dbutils.widgets.get("landing_root").strip().rstrip("/")
job_run_id = dbutils.widgets.get("job_run_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog identifier: {catalog!r}")

spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "load_dim_fx_rates"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
source_path = f"{landing_root}/dim_fx_rates/fx_rates.csv"
target_table = f"{catalog}.silver.dim_fx_rates"
audit_table = f"{catalog}.control.audit_pipeline_runs"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`silver`")

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`silver`.`dim_fx_rates` (
        rate_date DATE NOT NULL,
        currency STRING NOT NULL,
        rate_to_usd DECIMAL(20, 8) NOT NULL,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Daily currency conversion rates to USD'
    )
    """
)

In [ ]:
audit_schema = T.StructType(
    [
        T.StructField("pipeline_run_id", T.StringType(), False),
        T.StructField("pipeline_name", T.StringType(), False),
        T.StructField("batch_id", T.StringType(), True),
        T.StructField("start_ts", T.TimestampType(), False),
        T.StructField("end_ts", T.TimestampType(), True),
        T.StructField("status", T.StringType(), False),
        T.StructField("rows_read", T.LongType(), True),
        T.StructField("rows_written", T.LongType(), True),
        T.StructField("rows_quarantined", T.LongType(), True),
        T.StructField("source_paths", T.ArrayType(T.StringType()), True),
        T.StructField("notebook_job_run_id", T.StringType(), True),
        T.StructField("error_message", T.StringType(), True),
    ]
)


def append_audit(
    status: str,
    end_ts: datetime | None = None,
    rows_read: int | None = None,
    rows_written: int | None = None,
    error_message: str | None = None,
) -> None:
    audit_row = Row(
        pipeline_run_id=pipeline_run_id,
        pipeline_name=pipeline_name,
        batch_id=None,
        start_ts=start_ts,
        end_ts=end_ts,
        status=status,
        rows_read=rows_read,
        rows_written=rows_written,
        rows_quarantined=0,
        source_paths=[source_path],
        notebook_job_run_id=job_run_id,
        error_message=error_message,
    )
    audit_source = spark.createDataFrame([audit_row], audit_schema)
    audit_target = DeltaTable.forName(spark, audit_table)
    (
        audit_target.alias("target")
        .merge(
            audit_source.alias("source"),
            "target.pipeline_run_id = source.pipeline_run_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


append_audit("RUNNING")

In [ ]:
try:
    input_schema = T.StructType(
        [
            T.StructField("rate_date", T.StringType(), True),
            T.StructField("currency", T.StringType(), True),
            T.StructField("rate_to_usd", T.StringType(), True),
        ]
    )

    raw_fx = (
        spark.read.option("header", "true")
        .schema(input_schema)
        .csv(source_path)
        .select("*", F.col("_metadata.file_path").alias("source_file"))
    )

    rows_read = raw_fx.count()

    valid_fx = (
        raw_fx.select(
            F.to_date("rate_date").alias("rate_date"),
            F.upper(F.trim("currency")).alias("currency"),
            F.col("rate_to_usd").cast(T.DecimalType(20, 8)).alias("rate_to_usd"),
            F.current_timestamp().alias("ingest_ts"),
            "source_file",
            F.lit(pipeline_run_id).alias("pipeline_run_id"),
        )
        .filter(
            F.col("rate_date").isNotNull()
            & F.col("currency").isNotNull()
            & (F.length("currency") == 3)
            & F.col("rate_to_usd").isNotNull()
            & (F.col("rate_to_usd") > 0)
        )
        .dropDuplicates(["rate_date", "currency"])
    )

    rows_valid = valid_fx.count()
    if rows_valid != rows_read:
        raise ValueError(
            f"FX source contains invalid or duplicate rows: "
            f"rows_read={rows_read}, valid_unique_rows={rows_valid}"
        )

    target = DeltaTable.forName(spark, target_table)
    (
        target.alias("target")
        .merge(
            valid_fx.alias("source"),
            """
            target.rate_date = source.rate_date
            AND target.currency = source.currency
            """,
        )
        .whenMatchedUpdate(
            condition="target.rate_to_usd <> source.rate_to_usd",
            set={
                "rate_to_usd": "source.rate_to_usd",
                "ingest_ts": "source.ingest_ts",
                "source_file": "source.source_file",
                "pipeline_run_id": "source.pipeline_run_id",
            },
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    operation_metrics = target.history(1).select("operationMetrics").first()[
        "operationMetrics"
    ]
    rows_written = int(operation_metrics.get("numTargetRowsInserted", 0)) + int(
        operation_metrics.get("numTargetRowsUpdated", 0)
    )

    append_audit(
        status="SUCCESS",
        end_ts=datetime.now(timezone.utc),
        rows_read=rows_read,
        rows_written=rows_written,
    )
except Exception as exc:
    append_audit(
        status="FAILED",
        end_ts=datetime.now(timezone.utc),
        error_message=str(exc)[:4000],
    )
    raise

In [ ]:
spark.table(target_table).orderBy("rate_date", "currency").display()